# Bab 1 — Konvensi Umum dan Elemen API scikit-learn

**Buku:** *scikit-learn Cookbook, Third Edition* — John Sukup (Packt, 2025), Bab 1
**Sumber kode:** teks buku Bab 1 (halaman 3–10). Repositori resmi [PacktPublishing/Scikit-Learn-Cookbook-Third-Edition](https://github.com/PacktPublishing/Scikit-Learn-Cookbook-Third-Edition) tidak memiliki folder kode untuk bab ini.
**Direproduksi dan dijelaskan oleh:** Tarannia Raisha Putri — NIM 101032300231 — S1 Teknik Komputer, Telkom University

**Resep yang dibahas**

| # | Resep | Kode di buku? | Alat utama |
|---|--------|:---:|-----------|
| 1.1 | Filosofi desain scikit-learn | — | konsistensi, modularitas |
| 1.2 | Memahami *estimator* | ✔ | `LinearRegression`, `KMeans`, `fit`, `predict`, `fit_predict` |
| 1.3 | *Transformer* dan metode `transform()` | ✔ | `StandardScaler`, `fit`, `transform`, `fit_transform` |
| 1.4 | Menangani *estimator* dan *transformer* kustom | — | `BaseEstimator`, *mixin* |
| 1.5 | *Pipeline* dan otomatisasi alur kerja | — | `Pipeline`, MLOps |
| 1.6 | Atribut dan metode umum | ✔ | `coef_`, `intercept_`, `score` |
| 1.7 | *Tuning* hiperparameter dengan metode pencarian | ✔ | `set_params`, `get_params` |
| 1.8 | Bekerja dengan metadata: *tag* dan lainnya | — | *estimator tags*, *metadata routing* |
| 1.9 | Praktik terbaik penggunaan API | — | ringkasan praktik |

> Nomor resep ditambahkan untuk navigasi; buku tidak menomori resep. Bab 1 adalah bab konseptual: hanya resep 1.2, 1.3, 1.6, dan 1.7 yang memuat kode. Resep tanpa kode dibahas di sini sebagai teori saja, **tanpa menambahkan kode buatan sendiri** yang seolah-olah berasal dari buku.

> **Catatan tentang sumber kode.** Bab ini satu-satunya yang kodenya diambil dari teks buku, bukan dari repositori resmi (yang tidak menyediakan kode Bab 1). Setiap sel kode buku disalin dari buku tanpa mengubah logikanya; hanya komentarnya yang diterjemahkan. Di buku, setiap potongan kode diakhiri blok `# Output:` yang berisi hasil cetakan yang diharapkan. Blok itu **bukan kode** (sebagian bahkan tidak valid sebagai Python, misalnya `Coefficients: [0.95]`), jadi tidak disalin ke sel kode. Sebagai gantinya, output yang dicetak buku dikutip di bagian interpretasi dan dibandingkan dengan output hasil eksekusi yang sebenarnya.

## Gambaran Umum Bab

scikit-learn bisa dipelajari dengan cepat karena hampir semua kelasnya mengikuti kontrak yang sama. Model dilatih dengan `fit()`, membuat prediksi dengan `predict()`, dan alat pra-pemrosesan mengubah data dengan `transform()`. Hal yang dipelajari dari data disimpan sebagai atribut yang namanya diakhiri garis bawah (`coef_`, `mean_`), sedangkan pengaturan yang dipilih pengguna adalah hiperparameter yang dibaca dan diubah lewat `get_params()`/`set_params()`. Karena kontraknya seragam, satu model dapat ditukar dengan model lain tanpa menulis ulang kode di sekitarnya, dan komponen-komponen dapat dirangkai menjadi `Pipeline`.

Bab ini memperkenalkan kontrak tersebut dengan contoh-contoh yang sangat kecil (lima titik data, tiga baris). Dalam alur kerja ML, bab ini bukan satu tahap tertentu, melainkan "tata bahasa" yang dipakai oleh semua tahap di bab-bab berikutnya: pra-pemrosesan (Bab 2) memakai *transformer*, pemodelan (Bab 4–8) memakai *estimator*, dan *tuning* (Bab 4, 7, 8) memakai `get_params`/`set_params` di balik layar.

## Persiapan

In [1]:
# Pustaka inti yang dipakai di sepanjang bab ini
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

# Satu seed global agar setiap eksekusi notebook menghasilkan angka yang sama.
# (Sel-sel buku tetap memakai nilai random_state masing-masing.)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Catat lingkungan persis yang menghasilkan output di bawah
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")

Python       : 3.13.5
numpy        : 2.1.3
pandas       : 2.2.3
scikit-learn : 1.5.2
matplotlib   : 3.9.4


## Resep 1.1 — Filosofi Desain scikit-learn

**Tujuan.** Memahami prinsip-prinsip desain yang membuat API scikit-learn konsisten, sebelum melihat kodenya.

### Teori

**Intuisi.** scikit-learn dibangun di atas empat prinsip: **konsistensi**, **kesederhanaan**, **modularitas**, dan **dapat dipakai ulang**. Konsistensi berarti semua algoritme berbagi antarmuka yang sama, sehingga mengganti `LinearRegression` dengan `RandomForestRegressor` umumnya cukup dengan mengubah satu baris. Modularitas berarti setiap langkah (penskalaan, encoding, model) adalah objek terpisah yang dapat digabungkan, misalnya di dalam `Pipeline`, dan objek buatan sendiri dapat ikut serta selama mematuhi antarmuka yang sama.

**Formalisasi kontrak API.** Kontrak yang dibahas di bab ini dapat dirangkum sebagai berikut. Misalkan $X\in\mathbb{R}^{n\times p}$ adalah matriks fitur ($n$ sampel, $p$ fitur) dan $\mathbf{y}\in\mathbb{R}^n$ target (opsional).

| Peran | Metode | Arti matematis |
|---|---|---|
| *Estimator* (semua objek) | `fit(X, y=None)` | memperkirakan parameter $\hat\theta$ dari data latih |
| *Predictor* | `predict(X)` | menerapkan $\hat f_{\hat\theta}: \mathbb{R}^p\to\mathcal{Y}$ ke setiap baris |
| *Transformer* | `transform(X)` | menerapkan $T_{\hat\theta}: \mathbb{R}^p\to\mathbb{R}^{p'}$ ke setiap baris |
| *Model* | `score(X, y)` | mengembalikan satu angka kualitas (akurasi atau $R^2$) |

Hiperparameter $\lambda$ (dipilih pengguna, diberikan ke konstruktor) dipisahkan dengan tegas dari parameter $\hat\theta$ (dipelajari oleh `fit`, disimpan dalam atribut berakhiran `_`).

**Cara scikit-learn mengimplementasikannya.** Semua *estimator* mewarisi `sklearn.base.BaseEstimator`, yang menyediakan `get_params`/`set_params` dengan membaca argumen konstruktor. Kemampuan tambahan datang dari *mixin* seperti `ClassifierMixin`, `RegressorMixin` (yang menambahkan `score`), `TransformerMixin` (yang menambahkan `fit_transform`), dan `ClusterMixin` (yang menambahkan `fit_predict`). Konstruktor hanya menyimpan hiperparameter; semua perhitungan dilakukan di `fit`.

**Asumsi, kelebihan, keterbatasan.**

* Kelebihan: kurva belajar landai, kode mudah dibaca, dan semua alat (pipeline, *grid search*, *cross-validation*) bekerja dengan *estimator* apa pun.
* Asumsi: data berbentuk tabel numerik (array NumPy, DataFrame pandas, atau matriks *sparse*). Data mentah seperti teks atau gambar harus diubah menjadi fitur terlebih dahulu.
* Keterbatasan: antarmuka yang seragam tidak menyembunyikan perbedaan algoritmik. Model berbasis jarak tetap butuh penskalaan, dan model tertentu tetap tidak mendukung nilai hilang.

Buku tidak memberi kode untuk bagian ini. Catatan buku tentang ejaan: "scikit-learn" selalu ditulis dengan huruf kecil, dan "sci" adalah singkatan dari *science*.

### Poin Penting
* Empat prinsip desain: konsistensi, kesederhanaan, modularitas, dan dapat dipakai ulang.
* Hiperparameter (diberikan ke konstruktor) dipisahkan dari parameter hasil belajar (atribut berakhiran `_`).
* `BaseEstimator` dan *mixin* adalah mekanisme yang membuat semua kelas berbagi antarmuka yang sama.

## Resep 1.2 — Memahami *Estimator*

**Tujuan.** Melihat pola `fit()` → `predict()` pada model *supervised* (`LinearRegression`) dan jalan pintas `fit_predict()` pada model *unsupervised* (`KMeans`).

### Teori

**Intuisi.** *Estimator* adalah objek Python yang belajar dari data. `fit()` adalah tahap belajar, `predict()` adalah tahap memakai apa yang telah dipelajari pada data baru. Keduanya sengaja dipisahkan: pada masalah *supervised*, model di-fit pada data latih dan dipakai untuk memprediksi data lain (data uji atau data produksi). Pada *clustering*, sering kali yang diinginkan adalah label untuk data yang sama yang dipakai untuk belajar, sehingga tersedia jalan pintas `fit_predict()`.

**Matematika.**

* **Regresi linear** dengan satu fitur memodelkan $\hat y = \beta_0 + \beta_1 x$ dan memilih $\beta_0,\beta_1$ yang meminimalkan $\sum_i (y_i - \beta_0 - \beta_1 x_i)^2$. Solusinya $\hat\beta_1 = \dfrac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sum_i (x_i-\bar x)^2}$ dan $\hat\beta_0 = \bar y - \hat\beta_1 \bar x$ (dibahas mendalam di Bab 5).
* **K-means** dengan $K$ klaster memilih pusat $\boldsymbol\mu_1,\dots,\boldsymbol\mu_K$ dan penugasan yang meminimalkan *inertia* $\sum_i \min_k \lVert\mathbf{x}_i-\boldsymbol\mu_k\rVert^2$. Algoritme Lloyd bergantian antara (1) menugaskan setiap titik ke pusat terdekat dan (2) memindahkan setiap pusat ke rata-rata titik yang ditugaskan kepadanya, sampai tidak ada perubahan.

**Cara scikit-learn mengimplementasikannya.**

* `LinearRegression(fit_intercept=True)`: `fit` menyelesaikan *least squares* dan menyimpan `coef_` serta `intercept_`; `predict` menghitung `X @ coef_ + intercept_`.
* `KMeans(n_clusters=8, init="k-means++", n_init="auto", random_state=None)`: `fit` menjalankan algoritme Lloyd dari inisialisasi *k-means++* dan menyimpan `cluster_centers_`, `labels_`, dan `inertia_`. `fit_predict(X)` setara dengan `fit(X).labels_`. Karena `random_state=None`, inisialisasinya memakai generator acak global NumPy (yang diberi *seed* di sel persiapan).
* **Nomor klaster bersifat sembarang:** K-means tidak tahu klaster mana yang "pertama". Label `[0,0,0,1,1]` dan `[1,1,1,0,0]` menggambarkan partisi yang sama persis.

**Asumsi, kelebihan, keterbatasan.**

* `fit`/`predict` yang terpisah memungkinkan evaluasi yang jujur (latih di satu data, uji di data lain).
* `fit_predict` hanya masuk akal ketika label untuk data latih itu sendiri yang dicari. Ia tidak menghasilkan model yang dievaluasi pada data baru.
* Data contoh di sini sangat kecil (5 titik), jadi tujuannya menunjukkan pola API, bukan kinerja model.

### Kode

In [2]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Contoh data
X = np.array([[1], [2], [3], [4], [5]])  # Matriks fitur
y = np.array([1, 2, 3, 3.5, 5])  # Nilai target

# Buat dan fit model
model = LinearRegression()
model.fit(X, y)

# Prediksi nilai untuk data baru
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print(predictions)

[5.75 6.7 ]


In [3]:
# Garis yang dipelajari dan pengecekan manual prediksi (tambahan)
b1, b0 = model.coef_[0], model.intercept_
print(f"y_hat = {b0:.2f} + {b1:.2f} * x")
print("pengecekan manual untuk x = 6 dan 7:", [round(float(b0 + b1 * x), 2) for x in (6, 7)])

y_hat = 0.05 + 0.95 * x
pengecekan manual untuk x = 6 dan 7: [5.75, 6.7]


**Interpretasi output.** Model memprediksi **5.75** untuk $x=6$ dan **6.7** untuk $x=7$, sama dengan output yang dicetak buku (`[5.75, 6.7]`). Sel tambahan menunjukkan dari mana angka itu berasal: garis yang dipelajari adalah $\hat y = 0.05 + 0.95x$, dan $0.05 + 0.95\cdot6 = 5.75$, $0.05+0.95\cdot7=6.70$. Titik-titik baru berada di luar rentang latih ($x$ = 1–5), jadi ini adalah ekstrapolasi: model sekadar meneruskan garis lurus, dengan asumsi tren yang sama berlanjut.

In [4]:
# fit_predict tidak dipakai pada LinearRegression,
# tetapi sebagai contoh untuk clustering:
from sklearn.cluster import KMeans

# Contoh data
X = np.array([[1], [2], [3], [4], [5]])

# Contoh clustering KMeans
kmeans = KMeans(n_clusters=2)
labels = kmeans.fit_predict(X)
print(labels)

[0 0 0 1 1]


In [5]:
# Pusat klaster dan inertia yang dipelajari (tambahan)
print("pusat klaster:", kmeans.cluster_centers_.ravel())
print("inertia (jumlah kuadrat jarak ke pusat):", kmeans.inertia_)
print("fit_predict sama dengan labels_:", bool((labels == kmeans.labels_).all()))

pusat klaster: [2.  4.5]
inertia (jumlah kuadrat jarak ke pusat): 2.5
fit_predict sama dengan labels_: True


**Interpretasi output.** K-means membagi lima titik menjadi **{1, 2, 3}** (label 0) dan **{4, 5}** (label 1), sama dengan output buku (`[0,0,0,1,1]`). Sel tambahan memperlihatkan hasil yang dipelajari: pusat klaster di **2.0** dan **4.5** (rata-rata masing-masing kelompok), dengan *inertia* **2.5** $= (1-2)^2+0+(3-2)^2+(4-4.5)^2+(5-4.5)^2$. Titik 3 masuk ke kelompok kiri karena jaraknya ke 2.0 (yaitu 1) lebih kecil daripada ke 4.5 (yaitu 1.5). Bahwa label yang dicetak persis `[0 0 0 1 1]`, bukan `[1 1 1 0 0]`, bergantung pada inisialisasi acak; di sini hasilnya cocok dengan buku karena *seed* global di sel persiapan. Pengecekan juga mengonfirmasi bahwa `fit_predict` mengembalikan `labels_` yang sama dengan yang disimpan model.

### Poin Penting
* `fit()` belajar, `predict()` menerapkan; keduanya dipisah agar model dapat dievaluasi pada data yang tidak dilihat saat belajar.
* `LinearRegression` mempelajari $\hat y = 0.05 + 0.95x$ dan memprediksi 5.75 dan 6.7 untuk $x$ = 6 dan 7.
* `fit_predict()` pada `KMeans` mengembalikan label klaster untuk data latih itu sendiri; nomor klaster bersifat sembarang.

## Resep 1.3 — *Transformer* dan Metode `transform()`

**Tujuan.** Melihat pola `fit()` → `transform()` pada `StandardScaler`, beserta jalan pintasnya `fit_transform()`, dan memahami kapan masing-masing dipakai.

### Teori

**Intuisi.** *Transformer* adalah *estimator* yang tidak memprediksi target, melainkan mengubah data. `fit()` mempelajari apa yang diperlukan untuk transformasi (misalnya rata-rata dan simpangan baku setiap kolom), dan `transform()` menerapkannya. Pemisahan ini penting untuk mencegah kebocoran data: statistik dipelajari dari data latih saja, lalu **statistik yang sama** dipakai untuk mengubah data uji. Memanggil `fit` lagi pada data uji akan menghasilkan transformasi yang berbeda dan membuat evaluasi tidak jujur.

**Matematika.** `StandardScaler` mengubah setiap nilai menjadi **z-score**:
$$z_{ij} = \frac{x_{ij}-\mu_j}{\sigma_j},\qquad \mu_j=\frac1n\sum_i x_{ij},\qquad \sigma_j=\sqrt{\frac1n\sum_i (x_{ij}-\mu_j)^2}.$$
Simpangan baku yang dipakai adalah versi populasi (pembagi $n$, bukan $n-1$). Setelah transformasi, setiap kolom memiliki rata-rata 0 dan variansi 1.

**Cara scikit-learn mengimplementasikannya.**

* `StandardScaler(with_mean=True, with_std=True)`: `fit` menyimpan `mean_`, `var_`, dan `scale_` ($=\sigma_j$); `transform` menghitung `(X - mean_) / scale_`.
* `fit_transform(X)` berasal dari `TransformerMixin` dan setara dengan `fit(X).transform(X)` (beberapa *transformer* mengimplementasikannya dengan cara yang lebih efisien).
* Aturan praktis: `fit_transform` pada data latih, `transform` saja pada data uji. Di dalam `Pipeline`, aturan ini dijalankan secara otomatis.

**Asumsi, kelebihan, keterbatasan.**

* Standardisasi penting untuk model berbasis jarak, berbasis gradien, dan berpenalti (Bab 2, 4, 5–7), tetapi tidak berpengaruh pada model pohon (Bab 8).
* Rata-rata dan simpangan baku sensitif terhadap pencilan; `RobustScaler` adalah alternatif yang lebih tahan.
* Contoh di bawah melakukan `fit` dan `transform` pada data yang sama (tidak ada data uji), hanya untuk memperlihatkan mekanismenya.

### Kode

In [6]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Contoh data
X = np.array([[1, 2], [3, 4], [5, 6]])

# Buat instance StandardScaler
scaler = StandardScaler()

# Fit scaler pada data
scaler.fit(X)

# Transformasi data
X_scaled = scaler.transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


In [7]:
# Statistik yang dipelajari fit() dan pengecekan z-score secara manual (tambahan)
print("mean_ :", scaler.mean_, "| scale_ (simpangan baku populasi):", scaler.scale_.round(6))
print("z-score manual sama dengan transform():", bool(np.allclose((X - X.mean(axis=0)) / X.std(axis=0), X_scaled)))
print("rata-rata / simpangan baku tiap kolom setelah transformasi:", X_scaled.mean(axis=0), X_scaled.std(axis=0))

mean_ : [3. 4.] | scale_ (simpangan baku populasi): [1.632993 1.632993]
z-score manual sama dengan transform(): True
rata-rata / simpangan baku tiap kolom setelah transformasi: [0. 0.] [1. 1.]


**Interpretasi output.** Output-nya sama dengan buku: baris pertama menjadi **−1.22474487**, baris tengah **0**, dan baris terakhir **+1.22474487**, identik untuk kedua kolom. Sel tambahan menjelaskan alasannya. `fit()` mempelajari rata-rata **[3, 4]** dan simpangan baku **1.633** untuk kedua kolom ($\sqrt{((1-3)^2+0+(5-3)^2)/3}=\sqrt{8/3}\approx1.633$). Baris pertama berjarak −2 dari rata-rata, sehingga $z = -2/1.633 = -1.2247$, yaitu $-\sqrt{3/2}$. Kedua kolom memiliki pola jarak yang sama (selisih 2 antarbaris), sehingga nilai terstandardisasinya identik. Setelah transformasi, setiap kolom memiliki rata-rata 0 dan simpangan baku 1, sesuai definisi.

In [8]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Contoh data
X = np.array([[1, 2], [3, 4], [5, 6]])

# Buat instance StandardScaler dan
# lakukan fit_transform pada data dalam satu langkah
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


**Interpretasi output.** `fit_transform()` menghasilkan matriks yang persis sama dengan `fit()` lalu `transform()` di atas, juga sama dengan output buku. Perbedaannya bukan pada hasil, melainkan pada penggunaan: `fit_transform()` dipakai sekali pada data latih, sedangkan data uji hanya melewati `transform()` dengan `mean_` dan `scale_` yang sudah dipelajari, agar data uji diperlakukan dengan cara yang sama persis seperti data latih.

### Poin Penting
* *Transformer* mempelajari statistik di `fit()` dan menerapkannya di `transform()`; `fit_transform()` menggabungkan keduanya.
* `StandardScaler` menghasilkan z-score dengan simpangan baku populasi: di sini rata-rata [3, 4], skala 1.633, nilai ±1.2247.
* Lakukan `fit` hanya pada data latih; data uji hanya di-`transform` agar tidak terjadi kebocoran.

## Resep 1.4 — Menangani *Estimator* dan *Transformer* Kustom

**Tujuan.** Memahami cara membuat kelas sendiri yang dapat dipakai seperti *estimator* bawaan, misalnya di dalam `Pipeline` atau `GridSearchCV`.

### Teori

**Intuisi.** Karena semua alat scikit-learn hanya mengandalkan kontrak API (bukan kelas tertentu), kelas buatan sendiri dapat ikut dipakai selama mematuhi kontrak itu. Caranya adalah mewarisi `BaseEstimator` dan *mixin* yang sesuai, lalu mengimplementasikan `fit()` serta `transform()` (untuk *transformer*) atau `predict()` (untuk model).

**Aturan kontrak** yang harus dipenuhi kelas kustom:

1. `__init__` hanya menyimpan argumen sebagai atribut dengan nama yang sama, tanpa validasi atau perhitungan. Dari sinilah `get_params()` membaca hiperparameter, sehingga *grid search* dapat meng-*clone* dan mengubahnya.
2. `fit(X, y=None)` mempelajari parameter, menyimpannya dalam atribut berakhiran `_`, dan mengembalikan `self` (agar dapat dirangkai, misalnya `fit(X).transform(X)`).
3. `transform`/`predict` memeriksa bahwa objek sudah di-fit (`sklearn.utils.validation.check_is_fitted`) sebelum memakai atribut berakhiran `_`.

**Mixin** adalah kelas kecil yang menambahkan satu kemampuan tanpa menjadi induk utama: `TransformerMixin` memberi `fit_transform`, `RegressorMixin` memberi `score` berbasis $R^2$, `ClassifierMixin` memberi `score` berbasis akurasi. Dengan demikian kode yang sama tidak perlu ditulis ulang di setiap kelas.

**Asumsi, kelebihan, keterbatasan.** Kelas kustom memungkinkan logika khusus domain (misalnya fitur rasio atau pembersihan teks) masuk ke dalam *pipeline* dan ikut di-*tuning* serta divalidasi silang. Utilitas `sklearn.utils.estimator_checks.check_estimator` dapat menguji apakah sebuah kelas benar-benar mematuhi kontrak.

Buku tidak memberi kode untuk bagian ini dan menyebut bahwa contohnya dibahas di bab-bab selanjutnya.

### Poin Penting
* Kelas kustom mewarisi `BaseEstimator` + *mixin*, lalu mengimplementasikan `fit` dan `transform`/`predict`.
* `__init__` hanya menyimpan hiperparameter; `fit` menyimpan hasil belajar di atribut berakhiran `_` dan mengembalikan `self`.
* Dengan mematuhi kontrak ini, kelas kustom bekerja dengan `Pipeline`, `GridSearchCV`, dan *cross-validation*.

## Resep 1.5 — *Pipeline* dan Otomatisasi Alur Kerja

**Tujuan.** Memahami peran `Pipeline` dalam merangkai langkah-langkah ML dan kaitannya dengan praktik MLOps.

### Teori

**Intuisi.** Alur kerja ML pada dasarnya adalah urutan langkah: pra-pemrosesan, pelatihan, prediksi. `Pipeline` membungkus urutan itu menjadi satu objek sehingga setiap langkah selalu dijalankan dengan urutan yang benar, dan statistik pra-pemrosesan hanya dipelajari dari data latih.

**Matematika.** *Pipeline* dengan *transformer* $T_1,\dots,T_m$ dan *estimator* akhir $f$ adalah komposisi fungsi $\hat y = f(T_m(\cdots T_1(\mathbf{x})))$. Saat `fit`, setiap $T_k$ di-fit pada output $T_{k-1}$ dari data latih; saat `predict`, langkah-langkah yang sudah dipelajari hanya di-*transform* (rinciannya di Resep 2.4).

**Kaitan dengan MLOps.** Menurut buku, MLOps adalah praktik mengintegrasikan alur kerja ML ke dalam siklus hidup pengembangan dan operasi perangkat lunak: mengotomatiskan pengembangan, pengujian, *deployment*, dan pemeliharaan model agar tetap andal di produksi. Buku menyebut empat manfaatnya: menjembatani tim *data science*, *ML engineering*, dan operasi; memperbaiki kolaborasi; mempercepat *deployment* lewat otomatisasi dan reproduktibilitas; serta memperkuat pemantauan kinerja, *observability*, dan *explainability* (misalnya untuk mendeteksi *model drift*). Di scikit-learn, dukungan untuk MLOps datang dari `Pipeline` (otomatisasi pra-pemrosesan + model), `GridSearchCV` (optimasi hiperparameter), serta `joblib`/`pickle` untuk menyimpan model, dan model scikit-learn dapat diintegrasikan ke platform seperti MLflow atau Kubeflow.

**Asumsi, kelebihan, keterbatasan.** *Pipeline* membuat alur kerja dapat direproduksi dan bebas kebocoran, tetapi berbentuk rantai linear; percabangan per jenis kolom membutuhkan `ColumnTransformer` sebagai salah satu langkahnya.

Buku tidak memberi kode untuk bagian ini; *pipeline* dipraktikkan mulai Resep 2.4 dan dipakai di hampir semua bab sesudahnya.

### Poin Penting
* `Pipeline` merangkai *transformer* dan *estimator* menjadi satu objek dengan satu `fit`/`predict`.
* Merangkum alur kerja dalam satu objek mendukung reproduktibilitas, yang menjadi inti MLOps.
* Alat scikit-learn yang relevan untuk MLOps: `Pipeline`, `GridSearchCV`, `joblib`/`pickle`.

## Resep 1.6 — Atribut dan Metode Umum

**Tujuan.** Membaca parameter yang dipelajari model (`coef_`, `intercept_`) dan menilai kecocokan model dengan `score()`.

### Teori

**Intuisi.** Setelah `fit`, model menyimpan apa yang dipelajarinya dalam atribut berakhiran garis bawah. Untuk model linear, `coef_` berisi kemiringan (berapa perubahan prediksi per satuan kenaikan fitur) dan `intercept_` berisi nilai prediksi ketika semua fitur bernilai nol. Metode `score()` memberi satu angka kualitas: akurasi untuk *classifier* dan $R^2$ untuk regresor.

**Matematika.** Koefisien determinasi
$$R^2 = 1-\frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}$$
adalah proporsi variansi target yang dijelaskan model. Nilai 1 berarti fit sempurna dan 0 berarti tidak lebih baik daripada selalu memprediksi $\bar y$. Buku mencatat kritik terhadap $R^2$: nilainya tidak pernah turun ketika variabel ditambahkan, sehingga sering dipakai **adjusted $R^2$**, $\bar R^2 = 1-(1-R^2)\dfrac{n-1}{n-p-1}$, yang memberi penalti untuk jumlah variabel $p$.

**Cara scikit-learn mengimplementasikannya.** `LinearRegression.score(X, y)` berasal dari `RegressorMixin` dan memanggil `r2_score(y, self.predict(X))`. `coef_` berbentuk `(n_features,)` untuk satu target, dan `intercept_` berupa skalar.

**Asumsi, kelebihan, keterbatasan.** Contoh di bawah menilai model pada data yang sama dengan data untuk fitting, sehingga $R^2$-nya adalah ukuran kecocokan *in-sample*, bukan kemampuan generalisasi. Evaluasi yang jujur memerlukan data uji atau *cross-validation* (Bab 4).

### Kode

In [9]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Contoh data
X = np.array([[1], [2], [3], [4], [5]])  # Matriks fitur
y = np.array([1, 2, 3, 3.5, 5])  # Nilai target

# Buat dan fit model
model = LinearRegression()
model.fit(X, y)

# Akses koefisien (kemiringan model linear)
print("Coefficients:", model.coef_)

# Akses intersep sumbu y
print("Intercept:", model.intercept_)

# Gunakan metode score() untuk mengevaluasi model (nilai R-squared)
print("Model R-squared:", model.score(X, y))

Coefficients: [0.95]
Intercept: 0.050000000000000266
Model R-squared: 0.9809782608695652


In [10]:
# Hitung ulang koefisien dan R^2 dengan rumus tertutup (tambahan)
x = X.ravel()
b1 = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b0 = y.mean() - b1 * x.mean()
resid = y - (b0 + b1 * x)
r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"rumus tertutup -> kemiringan = {b1:.4f}, intersep = {b0:.4f}, R^2 = {r2:.6f}")
print("residual:", resid.round(2), "| SSE =", round((resid ** 2).sum(), 3), "| SST =", round(((y - y.mean()) ** 2).sum(), 3))
n, p = len(y), 1
print(f"adjusted R^2 = {1 - (1 - r2) * (n - 1) / (n - p - 1):.6f}")

rumus tertutup -> kemiringan = 0.9500, intersep = 0.0500, R^2 = 0.980978
residual: [-0.    0.05  0.1  -0.35  0.2 ] | SSE = 0.175 | SST = 9.2
adjusted R^2 = 0.974638


**Interpretasi output.**

* **Kemiringan 0.95** dan **R² 0.9809782608695652** sama persis dengan output buku. Setiap kenaikan satu satuan $x$ menaikkan prediksi sebesar 0.95, dan garis menjelaskan sekitar **98,1 %** variansi target.
* **Intersep** tercetak `0.050000000000000266`, sedangkan buku mencetak `0.04999999999999938`. Keduanya adalah 0.05 dengan galat pembulatan *floating-point* yang berbeda di digit ke-16 (sekitar $10^{-15}$), yang bergantung pada pustaka aljabar linear yang dipakai, bukan perbedaan hasil.
* Sel tambahan menghitung ulang semuanya dengan rumus tertutup dan mendapatkan angka yang sama. Residualnya hanya 0, 0.05, 0.1, −0.35, dan 0.2; jumlah kuadrat galatnya (SSE) 0.175 dibanding variansi total (SST) 9.2, sehingga $R^2 = 1-0.175/9.2 = 0.981$.
* *Adjusted* $R^2$ sedikit lebih rendah (**0.974638**) karena memberi penalti untuk satu variabel pada hanya lima titik data. Karena dinilai pada data latih, kedua angka ini mengukur kecocokan, bukan kemampuan prediksi pada data baru.

### Poin Penting
* Atribut berakhiran `_` (`coef_`, `intercept_`) menyimpan hasil belajar; di sini garisnya $\hat y = 0.05 + 0.95x$.
* `score()` pada regresor mengembalikan $R^2$ (di sini 0.981); pada *classifier* mengembalikan akurasi.
* Selisih intersep dengan buku hanya galat pembulatan di digit ke-16.
* $R^2$ pada data latih bersifat optimis; *adjusted* $R^2$ dan data uji memberi gambaran yang lebih jujur.

## Resep 1.7 — *Tuning* Hiperparameter dengan Metode Pencarian

**Tujuan.** Mengubah dan membaca hiperparameter model secara terprogram dengan `set_params()` dan `get_params()`, mekanisme yang dipakai oleh alat pencarian otomatis.

### Teori

**Intuisi.** Hiperparameter adalah pengaturan yang dipilih *sebelum* pelatihan (misalnya jumlah pohon atau kedalaman maksimum). scikit-learn menyediakan pencarian otomatis, yaitu `GridSearchCV`, `RandomizedSearchCV`, serta versi *successive halving*-nya (`HalvingGridSearchCV`, `HalvingRandomSearchCV`). Semua alat itu bekerja dengan cara yang sama di balik layar: meng-*clone* *estimator*, memanggil `set_params()` dengan kandidat pengaturan, lalu mem-fit dan menilai hasilnya. `get_params()` dan `set_params()` juga dapat dipanggil langsung untuk penyetelan manual.

**Formalisasi.** Sebuah *estimator* dengan hiperparameter $\lambda = (\lambda_1,\dots,\lambda_q)$ mendefinisikan keluarga model $\{\hat f_\lambda\}$. `get_params()` mengembalikan $\lambda$ sebagai *dictionary* `{nama: nilai}`, dan `set_params(**kw)` mengganti sebagian komponennya. Metode pencarian kemudian memilih $\hat\lambda = \arg\max_{\lambda\in\Lambda} \mathrm{CV}(\lambda)$ atas himpunan kandidat $\Lambda$ (dipraktikkan di Resep 4.3, 7.3, dan 8.4).

**Cara scikit-learn mengimplementasikannya.** `BaseEstimator.get_params(deep=True)` membaca nama argumen `__init__` dan nilai atribut dengan nama yang sama; untuk objek bersarang seperti `Pipeline`, parameter langkah ditampilkan dengan format `langkah__param`. `set_params` memvalidasi bahwa nama parameter ada, menetapkannya, lalu mengembalikan `self`. Mengubah hiperparameter setelah `fit` tidak memengaruhi model yang sudah dilatih sampai `fit` dipanggil lagi.

**Asumsi, kelebihan, keterbatasan.** `set_params` sangat praktis untuk eksperimen, tetapi tidak memilih nilai yang baik dengan sendirinya; itu tugas pencarian ber-*cross-validation*. Catatan: buku menyebut output `get_params()` sebagai hiperparameter yang memberi fit terbaik, padahal yang dicetak hanyalah pengaturan saat ini (nilai yang baru ditetapkan ditambah nilai default), karena model belum dilatih maupun di-*tuning*.

### Kode

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Buat model RandomForestClassifier
model = RandomForestClassifier()

# Tetapkan hiperparameter sebelum pelatihan dengan set_params()
model.set_params(n_estimators=100, max_depth=10, random_state=42)

# Periksa parameter yang sudah diperbarui
print(model.get_params())

{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [12]:
# Parameter mana yang benar-benar berbeda dari default? (tambahan)
defaults = RandomForestClassifier().get_params()
print("jumlah hiperparameter:", len(defaults))
print("berbeda dari default:", {k: (defaults[k], v) for k, v in model.get_params().items() if defaults[k] != v})
print("model sudah di-fit?", hasattr(model, "estimators_"))

jumlah hiperparameter: 19
berbeda dari default: {'max_depth': (None, 10), 'random_state': (None, 42)}
model sudah di-fit? False


**Interpretasi output.**

* *Dictionary* yang tercetak sama persis dengan output buku: 19 hiperparameter dengan `max_depth=10`, `n_estimators=100`, dan `random_state=42`, sedangkan sisanya bernilai default (misalnya `criterion='gini'`, `max_features='sqrt'`, `bootstrap=True`).
* Sel tambahan menunjukkan bahwa hanya **dua** pengaturan yang benar-benar berubah dari default, yaitu `max_depth` (None → 10) dan `random_state` (None → 42). `n_estimators=100` sudah merupakan nilai default sejak scikit-learn 0.22, jadi menetapkannya tidak mengubah apa pun.
* Model **belum di-fit** (belum ada atribut `estimators_`). Jadi output ini bukan "hiperparameter terbaik", melainkan konfigurasi yang akan dipakai ketika `fit` dipanggil. Memilih nilai terbaik membutuhkan `GridSearchCV` atau `RandomizedSearchCV`.

### Poin Penting
* `get_params()` membaca semua hiperparameter, `set_params()` mengubahnya dan mengembalikan objek itu sendiri.
* Alat pencarian (`GridSearchCV`, `RandomizedSearchCV`, versi *halving*) memakai `set_params` di balik layar untuk setiap kandidat.
* Di sini hanya `max_depth` dan `random_state` yang berubah dari default; output `get_params` adalah konfigurasi, bukan hasil *tuning*.

## Resep 1.8 — Bekerja dengan Metadata: *Tag* dan Lainnya

**Tujuan.** Mengenal dua jenis metadata di scikit-learn: *estimator tags* yang mendeskripsikan kemampuan *estimator*, dan *metadata routing* yang mengatur aliran informasi tambahan (seperti bobot sampel) di dalam *pipeline*.

### Teori

**Estimator tags.** Setiap *estimator* membawa *tag*, yaitu informasi tentang kemampuannya: misalnya apakah ia mendukung nilai hilang, keluaran multi-target, atau input non-negatif saja. scikit-learn memakai *tag* ini untuk menyesuaikan perilaku dalam *cross-validation*, *pipeline*, dan pengujian *estimator*. Kelas kustom (Resep 1.4) dapat mendeklarasikan *tag*-nya sendiri.

**Metadata routing.** Selain `X` dan `y`, sebagian langkah membutuhkan informasi tambahan, seperti `sample_weight` atau label kelompok (`groups`) untuk *cross-validation* berkelompok. *Metadata routing* mengatur ke mana informasi itu dikirim. Objek **router** (misalnya `Pipeline` atau `GridSearchCV`) meneruskan metadata, sedangkan objek **consumer** (misalnya sebuah *classifier*) memakainya dalam perhitungan. Setiap *consumer* menyatakan metadata apa yang dimintanya, misalnya dengan `set_fit_request(sample_weight=True)`, setelah fitur ini diaktifkan lewat `sklearn.set_config(enable_metadata_routing=True)`.

**Contoh dari buku (dalam kata-kata sendiri).** Pada dataset yang tidak seimbang, bobot sampel dapat dikirim hanya ke langkah yang membutuhkannya, misalnya *classifier* yang memakai pembobotan, dan tidak dikirim ke langkah penskalaan. Dengan begitu penanganan ketidakseimbangan tidak memengaruhi pra-pemrosesan.

**Asumsi, kelebihan, keterbatasan.** *Metadata routing* masih merupakan fitur yang relatif baru dan harus diaktifkan secara eksplisit di scikit-learn 1.5. Fitur ini membuat alur kerja lebih fleksibel, tetapi menambah konfigurasi yang harus dipahami.

Buku tidak memberi kode untuk bagian ini dan merujuk ke bab-bab tentang *cross-validation* dan *deployment* untuk contohnya.

### Poin Penting
* *Estimator tags* mendeskripsikan kemampuan *estimator* (misalnya dukungan nilai hilang atau multi-output).
* *Metadata routing* meneruskan informasi tambahan (bobot sampel, grup) dari *router* ke *consumer* yang memintanya.
* Di scikit-learn 1.5, *metadata routing* harus diaktifkan lewat `set_config(enable_metadata_routing=True)`.

## Resep 1.9 — Praktik Terbaik Penggunaan API

**Tujuan.** Merangkum kebiasaan yang membuat kode scikit-learn jelas, modular, dan mudah dipelihara.

### Teori

Buku menutup bab dengan lima praktik terbaik, di sini dirangkum beserta bagian buku yang mempraktikkannya:

| Praktik | Isi | Dipraktikkan di |
|---|---|---|
| API yang seragam | Pakai pola `fit()`, `transform()`, `predict()` yang sama untuk semua *estimator* | semua bab |
| Pra-pemrosesan data | Skalakan, encode, dan tangani nilai hilang dengan alat dari `sklearn.preprocessing` sebelum data masuk ke model | Bab 2 |
| *Pipeline* | Rangkai transformasi dan model dengan `Pipeline` agar kode ringkas dan *tuning* mudah | Resep 2.4, 2.6, 3.4 |
| *Cross-validation* | Nilai kinerja dengan teknik dari `sklearn.model_selection` untuk estimasi generalisasi yang andal | Bab 4 |
| *Tuning* hiperparameter | Cari hiperparameter dengan `GridSearchCV` atau `RandomizedSearchCV` | Resep 4.3, 7.3, 8.4 |

Prinsip di balik semuanya sama dengan Resep 1.1: komponen yang modular dan antarmuka yang konsisten memudahkan *debugging*, pengujian, dan penskalaan alur kerja ML. Buku tidak memberi kode untuk bagian ini.

### Poin Penting
* Lima kebiasaan inti: API seragam, pra-pemrosesan yang tepat, *pipeline*, *cross-validation*, dan *tuning* hiperparameter.
* Semuanya dipraktikkan secara konkret di Bab 2–8.

## Ringkasan Bab

| Resep | Konsep | Kelas/fungsi sklearn utama | Poin penting |
|---|---|---|---|
| 1.1 Filosofi desain | Konsistensi, kesederhanaan, modularitas, dapat dipakai ulang | `BaseEstimator`, *mixin* | Hiperparameter di konstruktor, hasil belajar di atribut berakhiran `_`. |
| 1.2 *Estimator* | `fit` → `predict`; `fit_predict` untuk *clustering* | `LinearRegression`, `KMeans` | Garis $\hat y = 0.05+0.95x$ memprediksi 5.75 dan 6.7; K-means memisahkan {1,2,3} dan {4,5}. |
| 1.3 *Transformer* | `fit` → `transform`; `fit_transform` | `StandardScaler` | z-score ±1.2247 dari rata-rata [3, 4] dan skala 1.633; `fit` hanya pada data latih. |
| 1.4 Kelas kustom | Kontrak API untuk kelas buatan sendiri | `BaseEstimator`, `TransformerMixin`, `check_is_fitted` | `__init__` hanya menyimpan, `fit` mengembalikan `self`. |
| 1.5 *Pipeline* | Merangkai langkah, MLOps | `Pipeline`, `joblib` | Satu objek untuk seluruh alur kerja mendukung reproduktibilitas. |
| 1.6 Atribut umum | `coef_`, `intercept_`, `score` | `LinearRegression` | Kemiringan 0.95, R² 0.981 (in-sample); intersep berbeda dari buku hanya di digit ke-16. |
| 1.7 Hiperparameter | `get_params`/`set_params` | `RandomForestClassifier` | Hanya `max_depth` dan `random_state` yang berubah; output adalah konfigurasi, bukan hasil *tuning*. |
| 1.8 Metadata | *Estimator tags*, *metadata routing* | `set_config(enable_metadata_routing=True)` | *Router* meneruskan metadata ke *consumer* yang memintanya. |
| 1.9 Praktik terbaik | Lima kebiasaan inti | `Pipeline`, `GridSearchCV`, `sklearn.preprocessing` | Dipraktikkan di Bab 2–8. |

**Kaitan dengan Bab 2.** Bab 1 memperkenalkan kontrak `fit`/`transform`/`predict` dengan contoh beberapa baris. Bab 2 memakai kontrak itu pada masalah nyata pertama dalam alur kerja ML: data yang **tidak lengkap, berbeda skala, dan bercampur kategori**. Imputer, scaler, dan encoder di sana adalah *transformer* seperti `StandardScaler` di Resep 1.3, dan semuanya dirangkai dengan `Pipeline` yang dibahas di Resep 1.5.

## Referensi

* Sukup, J. (2025). *scikit-learn Cookbook, Third Edition*. Packt Publishing. Bab 1, hlm. 1–12.
* scikit-learn User Guide — Getting Started (estimator, transformer, pipeline): <https://scikit-learn.org/1.5/getting_started.html>
* scikit-learn — Developing scikit-learn estimators (kontrak API, `BaseEstimator`, *mixin*, *tags*): <https://scikit-learn.org/1.5/developers/develop.html>
* scikit-learn User Guide — Metadata Routing: <https://scikit-learn.org/1.5/metadata_routing.html>
* scikit-learn User Guide — Model persistence: <https://scikit-learn.org/1.5/model_persistence.html>
* scikit-learn API — `StandardScaler`: <https://scikit-learn.org/1.5/modules/generated/sklearn.preprocessing.StandardScaler.html>
* Buitinck, L. et al. (2013). API design for machine learning software: experiences from the scikit-learn project. *ECML PKDD Workshop: Languages for Data Mining and Machine Learning*, 108–122.